# Breakout Room #1 — Palanca "r" vs AOV

**Objetivo:** comparar el valor de aumentar la retención (la palanca **r**) contra aumentar el
AOV (*Average Order Value*, el ticket promedio), usando la fórmula multiplicativa de CLV y los
datos reales de la base sintética de Starbucks Rewards.

```
CLV = (ARPU × margen) × r / (1 + d − r) − CAC
```

Trabajen las celdas en orden. Donde dice **`# TODO`** hay algo para que ustedes completen antes de
correr la celda. Al final van a tener la tabla y la gráfica que les pide la guía — y una respuesta
lista para defender: **¿quién gana, subir el AOV 10% o subir la retención de 90% a 95%?**


In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

transactions = pd.read_csv("/Volumes/workspace/default/semana3/transactions.csv", parse_dates=["transaction_date"])
customers = pd.read_csv("/Volumes/workspace/default/semana3/customers.csv", parse_dates=["signup_date", "last_active_date"])

print(f"{len(customers):,} clientes · {len(transactions):,} transacciones")
print(f"Rango de fechas: {transactions['transaction_date'].min().date()} a {transactions['transaction_date'].max().date()}")
transactions.head()


## Paso 1 — midan la retención mensual real

Un cliente está "activo" en un mes si tiene al menos una transacción ese mes. La retención
mes-a-mes es: de los clientes activos en el mes **M**, ¿qué fracción también compró en **M+1**?


In [0]:
transactions["month"] = transactions["transaction_date"].dt.to_period("M")

active_by_month = transactions.groupby("month")["customer_id"].apply(set).sort_index()
months = active_by_month.index.tolist()

retention_rows = []
for i in range(len(months) - 1):
    this_month, next_month = months[i], months[i + 1]
    active_this = active_by_month[this_month]
    active_next = active_by_month[next_month]
    retained = active_this & active_next
    retention_rows.append({
        "month": this_month,
        "active_customers": len(active_this),
        "retained_next_month": len(retained),
        "retention_rate": len(retained) / len(active_this),
    })

retention_df = pd.DataFrame(retention_rows).set_index("month")
retention_df.tail(12)


In [0]:
# TODO: usen los últimos 12 meses de la tabla de arriba para calcular una retención mensual promedio.
r_actual = retention_df["retention_rate"].tail(12).mean()
print(f"Retención mensual promedio (últimos 12 meses): {r_actual:.1%}")


## Paso 2 — midan el ARPU mensual real

ARPU = ingreso total del mes / clientes distintos que compraron ese mes (no el valor de una sola
orden: es cuánto deja, en promedio, cada cliente activo en un mes).


In [0]:
monthly = transactions.groupby("month").agg(
    revenue=("amount", "sum"),
    active_customers=("customer_id", "nunique"),
)
monthly["arpu"] = monthly["revenue"] / monthly["active_customers"]

arpu_actual = monthly["arpu"].tail(12).mean()
print(f"ARPU mensual promedio (últimos 12 meses): ${arpu_actual:.2f}")
monthly.tail(6)


In [0]:
# TODO: el CAC promedio ya está en customers.csv (columna acquisition_cost).
cac_actual = customers["acquisition_cost"].mean()
print(f"CAC promedio: ${cac_actual:.2f}")


## Paso 3 — la fórmula multiplicativa de CLV

Con margen = 70% y una tasa de descuento **mensual** d = 1%:


In [0]:
MARGIN = 0.70
D_MONTHLY = 0.01

def clv(arpu, r, margin=MARGIN, d=D_MONTHLY, cac=cac_actual):
    multiplier = r / (1 + d - r)
    return (arpu * margin) * multiplier - cac, multiplier

clv_base, mult_base = clv(arpu_actual, r_actual)
print(f"r = {r_actual:.1%}  →  multiplicador = {mult_base:.2f}×  →  CLV = ${clv_base:,.2f}")


## Paso 4 — reconstruyan la tabla del multiplicador (como la de la slide)

Un barrido de retenciones mensuales, de 70% a 97%, con el ARPU y el CAC que acaban de medir.


In [0]:
retention_grid = [0.70, 0.80, 0.85, 0.90, 0.95, 0.97]

table_rows = []
prev_mult = None
for r in retention_grid:
    clv_r, mult_r = clv(arpu_actual, r)
    change = None if prev_mult is None else (mult_r / prev_mult - 1)
    table_rows.append({
        "retencion_mensual": f"{r:.0%}",
        "multiplicador": round(mult_r, 2),
        "cambio_vs_anterior": None if change is None else f"{change:+.0%}",
        "CLV": round(clv_r, 2),
    })
    prev_mult = mult_r

multiplier_table = pd.DataFrame(table_rows)
multiplier_table


## Paso 5 — el experimento: AOV +10% vs retención 90% → 95%

Partan de una retención base de 90% (la más cercana a lo que midieron) y comparen, con el MISMO
CLV de partida, los dos escenarios de la guía.


In [0]:
R_BASE = 0.90
R_TARGET = 0.95
AOV_LIFT = 0.10

clv_start, _ = clv(arpu_actual, R_BASE)

# Escenario A: sube el ARPU 10%, la retención se queda en 90%
clv_aov_up, _ = clv(arpu_actual * (1 + AOV_LIFT), R_BASE)
lift_aov = clv_aov_up / clv_start - 1

# Escenario B: sube la retención de 90% a 95%, el ARPU no cambia
clv_r_up, _ = clv(arpu_actual, R_TARGET)
lift_r = clv_r_up / clv_start - 1

print(f"CLV de partida (r=90%):               ${clv_start:,.2f}")
print(f"Escenario A — AOV +10%:                ${clv_aov_up:,.2f}   ({lift_aov:+.0%})")
print(f"Escenario B — retención 90% → 95%:      ${clv_r_up:,.2f}   ({lift_r:+.0%})")
print()
ganador = "la retención (Escenario B)" if lift_r > lift_aov else "el AOV (Escenario A)"
print(f"Gana: {ganador}")


## Paso 6 — la gráfica

Dos vistas: el multiplicador no lineal a lo largo de la grilla de retención, y la comparación
directa de los dos escenarios.


In [0]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Izquierda: curva del multiplicador
axes[0].plot(multiplier_table["retencion_mensual"], multiplier_table["multiplicador"],
             marker="o", linewidth=3, color="#00704A")
axes[0].set_title("El multiplicador no es lineal")
axes[0].set_xlabel("Retención mensual")
axes[0].set_ylabel("Multiplicador  r / (1 + d − r)")
axes[0].grid(alpha=0.3)

# Derecha: comparación de los dos escenarios
scenarios = ["CLV base\n(r=90%)", "AOV +10%", "Retención\n90% → 95%"]
values = [clv_start, clv_aov_up, clv_r_up]
colors = ["#9A988E", "#11241C", "#00704A"]
bars = axes[1].bar(scenarios, values, color=colors)
axes[1].set_title("AOV +10% vs retención 90% → 95%")
axes[1].set_ylabel("CLV ($)")
for bar, v in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 5, f"${v:,.0f}",
                 ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/Volumes/workspace/default/semana3/clv_palanca_r_vs_aov.png", dpi=150)
plt.show()


## Para la plenaria

**1. ¿Cuál escenario subió más el CLV, y por cuánto?**
   
   El escenario que sube más el CLV es aumentar la retención de 90% -> 95%

**2. ¿Por qué el mismo esfuerzo (+10% en un número, +5 puntos en el otro) produce resultados tan
   distintos?**

   Aunque ambos parecen esfuerzos pequeños, no actúan igual. Subir el ticket promedio 10% significa que cada cliente nos deja aproximadamente 10% más dinero, pero sigue siendo el mismo cliente durante el mismo tiempo. En cambio, mejorar la retención de 90% a 95% hace que más clientes permanezcan comprando durante más meses, y cada mes adicional genera nuevas compras. Por eso el efecto se acumula y se multiplica en el tiempo.

**3. Si tuvieran que elegir UNA iniciativa para el próximo trimestre — una campaña de venta cruzada**
   (sube el AOV) o un programa de retención (sube r) — ¿con cuál se van, y por qué, visto el
   multiplicador?

   Escogeríamos aumentar la retención del 90% al 95% pues el efecto multiplicador al cambiar el % de retención permite incrementar considerablemente el CLV
